# Extracting BIOMASS Forest Height at In-Situ Plot Locations

Query the ESA MAAP STAC catalogue for BIOMASS Level-2a Forest Height
(`FP_FH__L2A`) products, stream them over authenticated HTTPS, and compute an
area-weighted canopy height for each subplot in the Forest Observation System
field dataset.

The catalogue is queried via the STAC API, which supports streaming, previewing
and downloading data. Products are read **without downloading**: only the pixel
window overlapping each plot crosses the network.

**Output:** a copy of the subplot table with three added columns — the mean
BIOMASS height per plot, the contributing pixel values, and the acquisition
times of the products used.


## 1. Setup

Connect to the catalogue, load the field data, and authenticate against ESA MAAP.
Everything here executes once and leaves state behind

### 1.1 Imports

In [98]:
%pip install aiohttp requests fsspec

import os
import math
import time
import pathlib

import numpy as np
import pandas as pd
import requests
import fsspec

import rasterio as rio
from rasterio.windows import from_bounds

from shapely import wkt
from pystac_client import Client

from tqdm import tqdm

from datetime import datetime

Note: you may need to restart the kernel to use updated packages.


### 1.2 Connect to the ESA MAAP STAC catalogue

In [99]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

### 1.3 Discover available BIOMASS collections

In [100]:
# find all Biomass collections
results = catalog.collection_search(
    filter="platform='Biomass'",
    datetime="2025-12-01T00:00:00.000Z/..",
)
print(f"{results.matched()} collections found.")

data = results.collection_list_as_dict()
df = pd.json_normalize(data, record_path=["collections"]).sort_values(
    by="title", ignore_index=True
)
df[["id", "title", "extent.temporal.interval"]]

17 collections found.


,id,title,extent.temporal.interval
0,BiomassAux,Biomass Auxiliary,"[[2024-07-31T00:00:00.000Z, None]]"
1,BiomassAuxIOC,Biomass Auxiliary (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
2,BiomassAuxRest,Biomass Auxiliary Restricted,"[[2024-07-31T00:00:00.000Z, None]]"
3,BiomassCalVal10,Biomass Cal/Val,"[[2025-04-29T00:00:00.000Z, None]]"
4,BiomassLevel0,Biomass Level 0,"[[2024-07-31T00:00:00.000Z, None]]"
5,BiomassLevel0IOC,Biomass Level 0 (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
6,BiomassLevel1a,Biomass Level 1A,"[[2025-11-21T00:00:00.000Z, None]]"
7,BiomassLevel1aIOC,Biomass Level 1A (IOC),"[[2024-07-31T00:00:00.000Z, None]]"
8,BiomassLevel1b,Biomass Level 1B,"[[2024-07-31T00:00:00.000Z, None]]"
9,BiomassLevel1bIOC,Biomass Level 1B (IOC),"[[2024-07-31T00:00:00.000Z, None]]"


### 1.4 Load in-situ field data (Forest Observation System)

In [101]:
# import csv with in-situ data (e.g., Forest Observation System data)

df = pd.read_csv("FOS_data_v2019.04.10.csv", sep=";")
df_geom = pd.read_csv("FOS_subplots_with_geometry.csv")

### 1.5 MAAP client credentials

In [102]:
# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

### 1.6 Exchange the offline token for an access token

In [103]:
if pathlib.Path("/home/jovyan/token.txt").exists(): # Put here your .txt file path
  with open("/home/jovyan/token.txt","rt") as f:
    OFFLINE_TOKEN = f.read().strip().replace("\n","")
    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')
    print("Access token retrieved successfully.")

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

Access token retrieved successfully.


## 2. Functions

Definitions only - running these cells computes nothing. Four helpers, in the
order the pipeline uses them: build a bounding box, search for products, inspect
a product, extract a weighted height.


### 2.1 Bounding box helper

In [104]:
# Supported area units, expressed as multipliers to square metres.
_AREA_UNITS = {
    "m2": 1.0,
    "sqm": 1.0,
    "ha": 1e4,
    "hectare": 1e4,
    "hectares": 1e4,
    "km2": 1e6,
    "sqkm": 1e6,
}

# WGS84 ellipsoid parameters
_A = 6378137.0            # semi-major axis, metres
_F = 1 / 298.257223563    # flattening
_E2 = _F * (2 - _F)       # first eccentricity squared


def bbox(lon, lat, area, unit="ha", crs="EPSG:4326", round_to=6):
    """
    Build a square bounding box of a given ground area centred on (lon, lat).

    Parameters
    ----------
    lon, lat : float
        Centroid coordinates, in the units of `crs`.
    area : float
        Desired ground area of the box.
    unit : str
        Area unit: 'ha', 'km2', 'm2', 'acre', ...
    crs : str
        CRS of the input/output coordinates. 'EPSG:4326' (geographic,
        degrees) is handled analytically; any projected metric CRS is
        handled directly in metres.
    round_to : int or None
        Decimal places for the returned values.

    Returns
    -------
    list
        [min_lon, min_lat, max_lon, max_lat]
    """
    key = unit.lower().replace(" ", "")
    if key not in _AREA_UNITS:
        raise ValueError(
            f"Unknown unit {unit!r}. Supported: {sorted(set(_AREA_UNITS))}"
        )
    if area <= 0:
        raise ValueError("area must be positive")

    area_m2 = area * _AREA_UNITS[key]
    half = math.sqrt(area_m2) / 2.0      # half-side of the square, metres

    geographic = str(crs).upper() in {"EPSG:4326", "4326", "WGS84", "CRS84"}

    if not geographic:
        # Projected CRS: units are already metres
        out = [lon - half, lat - half, lon + half, lat + half]
    else:
        if not (-90 <= lat <= 90):
            raise ValueError("lat must be within [-90, 90]")

        phi = math.radians(lat)
        sin_phi = math.sin(phi)
        w = math.sqrt(1 - _E2 * sin_phi ** 2)

        # Local radii of curvature on the WGS84 ellipsoid
        m_per_deg_lat = (math.pi / 180) * _A * (1 - _E2) / w ** 3
        m_per_deg_lon = (math.pi / 180) * _A * math.cos(phi) / w

        if m_per_deg_lon < 1e-6:          # at/near the poles
            raise ValueError("Longitude is degenerate at the poles.")

        dlat = half / m_per_deg_lat
        dlon = half / m_per_deg_lon

        min_lat = max(lat - dlat, -90.0)
        max_lat = min(lat + dlat, 90.0)

        min_lon = lon - dlon
        max_lon = lon + dlon
        if max_lon - min_lon >= 360:      # box wraps the whole globe
            min_lon, max_lon = -180.0, 180.0
        else:                             # normalise to [-180, 180]
            min_lon = (min_lon + 180) % 360 - 180
            max_lon = (max_lon + 180) % 360 - 180

        out = [min_lon, min_lat, max_lon, max_lat]

    return [round(v, round_to) for v in out] if round_to is not None else out
    

#### Get today's date

In [105]:
from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

### 2.2 Search the catalogue for products overlapping a plot

Given a row index, build a bounding box and query `BiomassLevel2a` for
`FP_FH__L2A` (Forest Height) products that intersect it.

Two routes to the bbox:

- **`geometry=True`** — parse the WKT polygon from the `geometry` column with
  `wkt.loads()` and take its `.bounds`. Preferred where subplot footprints exist.
- **`geometry=False`** — construct a square of `Plot_Area` centred on
  `Lon_cnt`/`Lat_cnt` using `bbox()`.

Both return `(min_lon, min_lat, max_lon, max_lat)`, so they are interchangeable
downstream.

Returns `(items, bb)` — the matching products and the bbox used, since the bbox
is needed again at extraction time for the area weighting.


In [106]:
# function to check if there are BIOMASS products at the location of a plot's bounding box

def search_catalogue (row, dataframe=df_geom, area_col_name="Plot_Area", lat_col_name="Lat_cnt", lon_col_name="Lon_cnt", unit="ha", geometry=False,
                      collection="BiomassLevel2a", product_filter="product:type='FP_FH__L2A'", time_to=today):

    if geometry:
        bb = wkt.loads(dataframe["geometry"].iloc[row]).bounds
    else:
        bb = bbox(dataframe[lon_col_name].iloc[row], dataframe[lat_col_name].iloc[row], area=dataframe[area_col_name].iloc[row], unit=unit, crs="EPSG:4326", round_to=6)

    search = catalog.search(
        collections = [collection],
        filter = product_filter, 
        datetime = ['2025-01-01T00:00:00Z',time_to],
        bbox = bb, 
        method = "GET",
        max_items = 100,  
    )
    
    items = list(search.items())
    print(f"Accessing {len(items)} items (limited by max_items).")
    print(f"{search.matched()} items found that matched the query.")

    return items, bb 
    
# example for retrieving single product's id
#i_product = search_catalogue(500, geometry=True)[0][0]
#i_bbox = search_catalogue(500, geometry=True)[1]
#print("\n", i_product, "\n\n", i_bbox)

### 2.3 Inspect a single product

Diagnostic helper. Prints a product's ID, acquisition datetime, footprint,
full property dictionary and — most### 2.3 Inspect a single product

Diagnostic helper. Prints a product's ID, acquisition datetime, footprint,
full property dictionary and — most usefully — its **asset keys and hrefs**.

Run this on any item to confirm the asset name before extraction. The Forest
Height raster is `enclosure_i_fh_tiff`, and the asset listing is also where to
look for any accompanying quality or validity layers.
 usefully — its **asset keys and hrefs**.

Run this on any item to confirm the asset name before extraction. The Forest
Height raster is `enclosure_i_fh_tiff`, and the asset listing is also where to
look for any accompanying quality or validity layers.


In [107]:
# function to inspect results, given a specific biomass product ID

def inspect_results(biomass_product):
    print(f"Item ID: {biomass_product.id}")
    print(f"  - Datetime: {biomass_product.datetime}")
    print(f"  - BBOX: {biomass_product.bbox}")
    print(f"  - Properties: {biomass_product.properties}")
    print("  - Assets:")
    for key, asset in biomass_product.assets.items():
        print(f"      • {key}: {asset.href}")
    print()

# example
# inspect_results(i_product)

### 2.4 Extract area-weighted forest height within a bounding box

Streams one product and returns `(weighted_mean, contributing_pixel_values)`.

**How it works**

1. `fsspec` opens the remote GeoTIFF with the bearer token attached in Python,
   and hands rasterio an already-open file object. Authenticating here rather
   than through GDAL's HTTP layer is what makes this work in this environment.
2. `from_bounds()` converts the geographic bbox to a pixel window;
   `.round_lengths().round_offsets()` snaps it to whole pixels so the window
   transform matches the array actually returned.
3. Only that window crosses the network — not the full scene.
4. Pixel edges are reconstructed from the window transform, and each pixel is
   weighted by the **area of its overlap with the bbox**. A pixel half inside
   contributes half as much as one fully inside.
5. Nodata and non-finite pixels get zero weight, so fill values are excluded
   from both numerator and denominator rather than dragged into the mean.

Returns `(nan, [])` where no valid pixels overlap — expected, since BIOMASS swath
products leave empty corners inside their north-up bounding rectangles.

Units: metres. Single band (for BIOMASS height product).


In [108]:
def height_in_bbox_full(product, bb, token, nodata=-9999):
    url = product.assets['enclosure_i_fh_tiff'].href

    fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"})
    f = fs.open(url, "rb")
    ds = rio.open(f)
    try:
        win = from_bounds(*bb, transform=ds.transform).round_lengths().round_offsets()
        arr = ds.read(1, window=win, boundless=True, fill_value=nodata).astype(float)
        T = ds.window_transform(win)
    finally:
        try: ds.close()
        except Exception: pass
        try: f.close()
        except Exception: pass

    px, py = T.a, abs(T.e)
    xs = T.c + np.arange(arr.shape[1] + 1) * px
    ys = T.f - np.arange(arr.shape[0] + 1) * py

    wx = np.clip(np.minimum(xs[1:], bb[2]) - np.maximum(xs[:-1], bb[0]), 0, None)
    wy = np.clip(np.minimum(ys[:-1], bb[3]) - np.maximum(ys[1:], bb[1]), 0, None)
    w = np.outer(wy, wx)

    w[(arr == nodata) | ~np.isfinite(arr)] = 0

    if w.sum() == 0:
        return np.nan, []

    mean = float((arr * w).sum() / w.sum())
    pixels = [round(float(v), 3) for v in arr[w > 0]]   # only contributing pixels

    return mean, pixels

# example
#height_in_bbox(i_product, i_bbox, access_token)
#print(df_geom.iloc[500][["Plot_ID", "H_Lorey_local", "Year_Census"]])


## 3. Batch extraction across all plots

For each row: search for overlapping products, extract a weighted mean from each,
and write three columns to a copy of `df_geom`.

| Column | Contents |
|---|---|
| `average_BIOMASS_height_in_plot` | Mean across all products found, metres |
| `BIOMASS_pixels_values` | List of lists — contributing pixels, one inner list per product |
| `BIOMASS_products_acquisition_time` | Acquisition timestamps, one per product |

Rows with no matching products stay `NaN`. Per-row exceptions are caught and
logged so one bad product cannot abort the run.

`contextlib.redirect_stdout` suppresses the two print statements inside
`search_catalogue()`, which would otherwise emit thousands of lines.

**On the list columns.** CSV stores these as text. Recover with
`ast.literal_eval` on read, or use `to_parquet()` to preserve them natively.


In [ ]:
import numpy as np, pandas as pd, contextlib, io
from tqdm import tqdm

df_results = df_geom.copy()
df_results["average_BIOMASS_height_in_plot"] = np.nan
df_results["BIOMASS_pixels_values"] = ""
df_results["BIOMASS_products_acquisition_time"] = ""

pbar = tqdm(range(len(df_results)))

for row in pbar:

    try:
        with contextlib.redirect_stdout(io.StringIO()):
            items, bb = search_catalogue(row, dataframe=df_results, geometry=True)
    except Exception:
        pbar.set_description(f"row {row}: search failed")
        continue

    if not items:
        pbar.set_description(f"row {row}: no products")
        continue

    heights, pixel_lists, times = [], [], []

    for it in items:
        try:
            h, px_vals = height_in_bbox_full(it, bb, access_token)
        except Exception:
            continue

        if h is not None and np.isfinite(h):
            heights.append(h)
            pixel_lists.append(px_vals)
            times.append(it.properties.get("datetime", ""))

    if heights:
        idx = df_results.index[row]
        df_results.loc[idx, "average_BIOMASS_height_in_plot"] = np.mean(heights)
        df_results.loc[idx, "BIOMASS_pixels_values"] = str(pixel_lists)
        df_results.loc[idx, "BIOMASS_products_acquisition_time"] = str(times)
        pbar.set_description(f"row {row}: {len(heights)} prod, {np.mean(heights):.1f} m")
    else:
        pbar.set_description(f"row {row}: no valid pixels")

df_results.to_csv("plots_with_biomass_height_and_additional_info.csv", index=False)


row 13: 2 prod, 12.2 m:   2%|▏         | 14/688 [00:45<34:17,  3.05s/it] 